In [97]:
import os
import pandas as pd
import numpy as np
import datetime
from scipy import sparse

## Download Kaggle Dataset

In [98]:
"""!pip install kaggle --quiet"""

'!pip install kaggle --quiet'

In [99]:
"""import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')"""

"import os\nfrom google.colab import userdata\nos.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')\nos.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')"

In [100]:
"""import kagglehub
import shutil

# 1. Download the dataset using kagglehub
# Replace with the dataset you want
dataset_handle = "kartik2112/fraud-detection"
cache_path = kagglehub.dataset_download(dataset_handle)

drive_path = "/content/drive/MyDrive/Math168/fraud_detection"

# 3. Create the directory if it doesn't exist
if not os.path.exists(drive_path):
    os.makedirs(drive_path)
    print(f"Created directory: {drive_path}")

# 4. Move (or Copy) the files
# shutil.copytree is best for moving entire folders
# We use dirs_exist_ok=True to avoid errors if you run this twice
shutil.copytree(cache_path, drive_path, dirs_exist_ok=True)

print(f"✅ Success! Files moved from cache to: {drive_path}")
print("Files in Drive:", os.listdir(drive_path))"""

'import kagglehub\nimport shutil\n\n# 1. Download the dataset using kagglehub\n# Replace with the dataset you want\ndataset_handle = "kartik2112/fraud-detection"\ncache_path = kagglehub.dataset_download(dataset_handle)\n\ndrive_path = "/content/drive/MyDrive/Math168/fraud_detection"\n\n# 3. Create the directory if it doesn\'t exist\nif not os.path.exists(drive_path):\n    os.makedirs(drive_path)\n    print(f"Created directory: {drive_path}")\n\n# 4. Move (or Copy) the files\n# shutil.copytree is best for moving entire folders\n# We use dirs_exist_ok=True to avoid errors if you run this twice\nshutil.copytree(cache_path, drive_path, dirs_exist_ok=True)\n\nprint(f"✅ Success! Files moved from cache to: {drive_path}")\nprint("Files in Drive:", os.listdir(drive_path))'

## Load Training Dataset

In [101]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [102]:
path = "/content/drive/MyDrive/Math168/fraud_detection/"
print(os.listdir(path))

['fraudTrain.csv', 'fraudTest.csv', 'tx_birank_priors.csv', 'feat_train.csv', 'feat_test1m.csv', 'tx_birank_priors_train.csv', 'tx_birank_priors_test1m.csv', 'card_birank.csv', 'merchant_birank.csv', 'apate_features.csv', 'apate_features_birank.csv']


In [103]:
fraudTest = pd.read_csv(path + "fraudTest.csv", index_col=0)
fraudTrain = pd.read_csv(path + "fraudTrain.csv", index_col=0)

In [104]:
fraudTrain['trans_date_trans_time'] = pd.to_datetime(fraudTrain['trans_date_trans_time'], errors='coerce')
fraudTest['trans_date_trans_time'] = pd.to_datetime(fraudTest['trans_date_trans_time'], errors='coerce')

In [105]:
fraudTrain["unix_time"] = (fraudTrain["trans_date_trans_time"].view("int64") // 10**9).astype("int64")
fraudTest['unix_time'] = (fraudTest["trans_date_trans_time"].view("int64") // 10**9).astype("int64")

/tmp/ipykernel_212/3512795679.py:1: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  fraudTrain["unix_time"] = (fraudTrain["trans_date_trans_time"].view("int64") // 10**9).astype("int64")
/tmp/ipykernel_212/3512795679.py:2: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  fraudTest['unix_time'] = (fraudTest["trans_date_trans_time"].view("int64") // 10**9).astype("int64")


In [106]:
test_start = fraudTest['trans_date_trans_time'].min()
test_end = test_start + pd.DateOffset(months=1)   # first calendar month
fraudTest_1m = fraudTest[
    (fraudTest['trans_date_trans_time'] >= test_start) &
    (fraudTest['trans_date_trans_time'] < test_end)
]

print("fraudTrain rows:", len(fraudTrain))
print("fraudTest total rows:", len(fraudTest))
print("fraudTest first-month rows:", len(fraudTest_1m))
print("Test window:", test_start, "to", test_end)

fraudTrain rows: 1296675
fraudTest total rows: 555719
fraudTest first-month rows: 87645
Test window: 2020-06-21 12:14:25 to 2020-07-21 12:14:25


In [107]:
fraudTrain = fraudTrain.assign(_split="train")
fraudTest_1m = fraudTest_1m.assign(_split="test_1m")

In [108]:
fraudAll = pd.concat([fraudTrain, fraudTest_1m], ignore_index=True)
fraudAll["row_id_combined"] = np.arange(len(fraudAll), dtype=np.int64)

In [109]:
fraudAll

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,_split,row_id_combined
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1546300818,36.011293,-82.048315,0,train,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1546300844,49.159047,-118.186462,0,train,1
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1546300851,43.150704,-112.154481,0,train,2
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1546300876,47.034331,-112.561071,0,train,3
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1546300986,38.674999,-78.632459,0,train,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1384315,2020-07-21 12:13:27,6011366578560244,"fraud_Huel, Hammes and Witting",grocery_pos,102.10,Adam,Stark,M,0912 Mark Fields Apt. 080,Mc Veytown,...,4653,Nutritional therapist,1997-07-01,28dbfcfd6288b406e125154d26a8048b,1595333607,41.103110,-77.601875,0,test_1m,1384315
1384316,2020-07-21 12:13:30,2264937662466770,fraud_Labadie LLC,personal_care,7.95,Juan,Sherman,M,5939 Garcia Forges Suite 297,San Antonio,...,1595797,Land,1995-10-17,a057b7eecf1683d1109afd59d6478d6a,1595333610,29.993934,-98.163099,0,test_1m,1384316
1384317,2020-07-21 12:13:32,30082025922891,fraud_Ernser-Feest,home,123.11,Kathleen,Thompson,F,199 Patterson Fords Apt. 132,Naples,...,276002,"Pilot, airline",1934-06-23,ba8f31627906ee914d3150b3557ebe03,1595333612,25.973459,-81.848113,0,test_1m,1384317
1384318,2020-07-21 12:13:36,30235438713303,"fraud_Kerluke, Considine and Macejkovic",misc_net,811.45,James,Baldwin,M,3603 Mitchell Court,Winfield,...,5512,Exhibition designer,1980-03-24,014019287afe680e698414b3ca38dc09,1595333616,37.583384,-82.006009,0,test_1m,1384318


### Load Birank Values

In [110]:
card_birank_daily_df = pd.read_csv(path + "card_birank.csv")
merchant_birank_daily_df = pd.read_csv(path + "merchant_birank.csv")

card_birank_daily_df["target_day"] = pd.to_datetime(card_birank_daily_df["target_day"]).dt.floor("D")
merchant_birank_daily_df["target_day"] = pd.to_datetime(merchant_birank_daily_df["target_day"]).dt.floor("D")

card_birank_daily_df["cc_num"] = card_birank_daily_df["cc_num"].astype(str)
merchant_birank_daily_df["merchant"] = merchant_birank_daily_df["merchant"].astype(str)

card_day_to_dict = {
    d: dict(zip(g["cc_num"].to_numpy(), g["card_birank"].to_numpy()))
    for d, g in card_birank_daily_df.groupby("target_day", sort=False)
}
merchant_day_to_dict = {
    d: dict(zip(g["merchant"].to_numpy(), g["merchant_birank"].to_numpy()))
    for d, g in merchant_birank_daily_df.groupby("target_day", sort=False)
}

## Build Bipartite (Tripartite) Graph

In [111]:
def get_snapshot(trans, snapshot_time: datetime, gamma, lookback_days, labeling_delay_days):
  snapshot_unix = snapshot_time.timestamp()

  lookback_sec = lookback_days * 24 * 60 * 60
  delay_sec = labeling_delay_days * 24 * 60 * 60

  history = trans[(trans['unix_time'] < snapshot_unix) & (
      trans['unix_time'] >= (snapshot_unix - lookback_sec))]

  confirmed_time = snapshot_unix - delay_sec
  known_fraud = history[(history['unix_time'] < confirmed_time) &
                          (history['is_fraud'] == 1)].copy()
  h = (snapshot_unix - known_fraud['unix_time'])/3600
  known_fraud['starting_risk'] = np.exp(-gamma * h)
  return history, known_fraud

In [112]:
def elapsed_in_unit(delta_seconds, time_unit):
    if time_unit == "minutes":
        return delta_seconds / 60.0
    elif time_unit == "hours":
        return delta_seconds / 3600.0
    elif time_unit == "days":
        return delta_seconds / 86400.0
    else:
        raise ValueError(f"Unknown time_unit: {time_unit}")

In [113]:
def get_snapshot_fast(trans, u, snapshot_time, gamma, lookback_days, labeling_delay_days, time_unit):
    snapshot_unix = snapshot_time.timestamp()
    lookback_sec = lookback_days * 86400
    delay_sec = labeling_delay_days * 86400

    left_time = snapshot_unix - lookback_sec
    right_time = snapshot_unix

    # index range for history
    left_idx = np.searchsorted(u, left_time, side="left")
    right_idx = np.searchsorted(u, right_time, side="left")  # unix_time < snapshot_unix
    history = trans.iloc[left_idx:right_idx]

    confirmed_time = snapshot_unix - delay_sec
    confirmed_idx = np.searchsorted(u, confirmed_time, side="left")

    # known fraud must be within history and earlier than confirmed_time
    # so it's within [left_idx, min(right_idx, confirmed_idx))
    k_right = min(right_idx, confirmed_idx)
    known_slice = trans.iloc[left_idx:k_right]

    # filter frauds only (vectorized)
    known_fraud = known_slice[known_slice["is_fraud"] == 1].copy()

    if not known_fraud.empty:
        delta_sec = snapshot_unix - known_fraud["unix_time"].to_numpy()
        t = elapsed_in_unit(delta_sec, time_unit)
        known_fraud["starting_risk"] = np.exp(-gamma * t)
    else:
        known_fraud["starting_risk"] = np.array([], dtype=float)

    return history, known_fraud

In [114]:
"""date_string = "2019-06-25"
snapshot_dt = pd.to_datetime(date_string)
hist, seeds = get_snapshot(fraudTrain, snapshot_dt, 0.03, 7, 1)"""

'date_string = "2019-06-25"\nsnapshot_dt = pd.to_datetime(date_string)\nhist, seeds = get_snapshot(fraudTrain, snapshot_dt, 0.03, 7, 1)'

In [115]:
def build_tripartite_matrix(history, snapshot_time, gamma, time_unit):
    # Extract columns as arrays
    cc = history["cc_num"].to_numpy()
    mc = history["merchant"].to_numpy()
    tx = history["trans_num"].to_numpy()

    # Fast factorization: get integer codes + categories
    cc_cat = pd.Categorical(cc)
    mc_cat = pd.Categorical(mc)
    tx_cat = pd.Categorical(tx)

    c_codes = cc_cat.codes.astype(np.int32)
    m_codes_local = mc_cat.codes.astype(np.int32)
    t_codes_local = tx_cat.codes.astype(np.int32)

    n_c = len(cc_cat.categories)
    n_m = len(mc_cat.categories)
    n_t = len(tx_cat.categories)

    m_codes = m_codes_local + n_c
    t_codes = t_codes_local + n_c + n_m

    # Build maps (string -> node idx)
    c_map = {k: i for i, k in enumerate(cc_cat.categories)}
    m_map = {k: i + n_c for i, k in enumerate(mc_cat.categories)}
    t_map = {k: i + n_c + n_m for i, k in enumerate(tx_cat.categories)}

    # Compute weights vectorized
    delta_sec = snapshot_time.timestamp() - history["unix_time"].to_numpy()
    t = elapsed_in_unit(delta_sec, time_unit)
    w = np.exp(-gamma * t).astype(np.float64)

    n = len(history)
    # 4 directed edges per transaction row
    row = np.empty(4 * n, dtype=np.int32)
    col = np.empty(4 * n, dtype=np.int32)
    data = np.empty(4 * n, dtype=np.float64)

    # Fill slices: (c<->t) and (m<->t)
    # c -> t
    row[0:n] = c_codes
    col[0:n] = t_codes
    data[0:n] = w

    # t -> c
    row[n:2*n] = t_codes
    col[n:2*n] = c_codes
    data[n:2*n] = w

    # m -> t
    row[2*n:3*n] = m_codes
    col[2*n:3*n] = t_codes
    data[2*n:3*n] = w

    # t -> m
    row[3*n:4*n] = t_codes
    col[3*n:4*n] = m_codes
    data[3*n:4*n] = w

    matrix_size = n_c + n_m + n_t
    Q = sparse.csr_matrix((data, (row, col)), shape=(matrix_size, matrix_size))
    return Q, c_map, m_map, t_map, w

In [116]:
def get_starting_vector(known_fraud, c_map, m_map, t_map):
  total_nodes = len(c_map) + len(m_map) + len(t_map)
  z = np.zeros(total_nodes)

  if known_fraud.empty:
        return z

  t_ids = known_fraud["trans_num"].to_numpy()
  risks = known_fraud["starting_risk"].to_numpy()

  idx = np.array([t_map.get(t, -1) for t in t_ids], dtype=np.int32) # if t_id exist in my t_mapping
  mask = idx >= 0
  z[idx[mask]] = risks[mask]

  s = z.sum()
  if s > 0:
      z /= s
  return z

In [117]:
def get_starting_vector_add_birank(
    known_fraud, c_map, m_map, t_map,
    snapshot_time,
    card_day_to_dict, merchant_day_to_dict
):
    total_nodes = len(c_map) + len(m_map) + len(t_map)
    z = np.zeros(total_nodes, dtype=np.float64)

    if known_fraud is not None and not known_fraud.empty:
        t_ids = known_fraud["trans_num"].to_numpy()
        risks = known_fraud["starting_risk"].to_numpy(dtype=float)

        idx = np.array([t_map.get(t, -1) for t in t_ids], dtype=np.int32)
        mask = idx >= 0
        z[idx[mask]] = risks[mask]

    day = pd.Timestamp(snapshot_time).floor("D")
    card_scores = card_day_to_dict.get(day, {})
    merch_scores = merchant_day_to_dict.get(day, {})

    for cc, i in c_map.items():
        z[i] += float(card_scores.get(str(cc), 0.0))

    for m, j in m_map.items():
        z[j] += float(merch_scores.get(str(m), 0.0))

    # 3) Normalize once (only if nonzero)
    s = z.sum()
    if s > 0:
        z /= s
    return z

In [118]:
def normalize(Q):
  col_sums = np.array(Q.sum(axis=0)).flatten()
  col_sums[col_sums == 0] = 1 # prevent division by zero
  inv_col_sums = sparse.diags(1.0 / col_sums)

  Q_norm = Q.dot(inv_col_sums)
  return Q_norm

In [119]:
def iteration(Q_norm, z_norm, alpha, max_iter = 5000, tol = 1e-6, check_every=10):
  # xi = np.random.rand(Q_norm.shape[0])
  xi = z_norm.copy()
  for i in range(int(max_iter)):
    prev_xi = xi
    xi = alpha * Q_norm.dot(prev_xi) + (1-alpha) * z_norm
    if (i + 1) % check_every == 0:
            err = np.linalg.norm(xi - prev_xi)
            # err = np.max(np.abs(xi - prev_xi))
            if err < tol:
                print(f"Converged after {i+1} iterations.")
                break
  return xi


In [120]:
def build_txscore_support_from_weights(history, w, t_scores):
  cc_wsum = pd.Series(w).groupby(history["cc_num"].to_numpy()).sum()
  mc_wsum = pd.Series(w).groupby(history["merchant"].to_numpy()).sum()

  # Latest historical transaction for each (cc, merchant) pair
  # If history is already time-sorted, drop_duplicates(..., keep="last") gives most recent
  pair_latest = history[["cc_num", "merchant", "trans_num"]].drop_duplicates(
      ["cc_num", "merchant"], keep="last"
  )
  pair_latest["tx_score_latest"] = pair_latest["trans_num"].map(t_scores)

  pair_latest_score = pair_latest.set_index(["cc_num", "merchant"])["tx_score_latest"]
  # latest tx score for a given transaction that existed between (card, merchant)

  return cc_wsum, mc_wsum, pair_latest_score

In [121]:
def compute_apate_txscore_only(
    today_txns, c_col, m_col, cc_wsum, mc_wsum, pair_latest_score
):
    cc_den = today_txns["cc_num"].map(cc_wsum).fillna(0.0).to_numpy(dtype=np.float64) + 1.0
    mc_den = today_txns["merchant"].map(mc_wsum).fillna(0.0).to_numpy(dtype=np.float64) + 1.0

    tx_col = c_col / cc_den + m_col / mc_den

    # direct score override (no trans_num intermediate)
    pair_index = pd.MultiIndex.from_arrays(
        [today_txns["cc_num"].to_numpy(), today_txns["merchant"].to_numpy()]
    ) # pair keys of (card, merchant) in today's transaction
    override_scores = pair_latest_score.reindex(pair_index) # fetch score if exists

    mask = ~override_scores.isna().to_numpy() # True if can be overritten
    if mask.any():
        tx_col[mask] = override_scores.to_numpy(dtype=np.float64)[mask]

    return tx_col

In [122]:
def main_network_featurization(trans_df, u, snapshot_time, z_mode="apate", # or apate_birank
    card_day_to_dict=None,
    merchant_day_to_dict=None):
    snapshot_time = pd.to_datetime(snapshot_time)

    time_windows = {
        'ST': {'gamma': 0.03, 'lookback': 1, 'delay': 0.5, 'time_unit': 'minutes'},
        'MT': {'gamma': 0.004, 'lookback': 7, 'delay': 3, 'time_unit': 'hours'},
        'LT': {'gamma': 0.0001, 'lookback': 30, 'delay': 7, 'time_unit': 'days'}
    }

    # slice today's txns ONCE (not per window)
    start = snapshot_time.normalize() # start at midnight
    end = start + pd.Timedelta(days=1)
    today_txns = trans_df[(trans_df["_dt"] >= start) & (trans_df["_dt"] < end)]

    # if no txns today, return empty
    if today_txns.empty:
        return pd.DataFrame(columns=["trans_num"] + [
            f"{k}_{w}" for w in time_windows for k in ("CCHScore", "MCScore", "TXScore")
        ])

    result = pd.DataFrame({"trans_num": today_txns["trans_num"].to_numpy()})

    for window_name, params in time_windows.items():
        print(f"Processing {window_name} network features...")
        history, known_fraud = get_snapshot_fast(
            trans_df,
            u,
            snapshot_time,
            params['gamma'],
            params['lookback'],
            params['delay'],
            params['time_unit']
        )

        Q, c_map, m_map, t_map, row_w = build_tripartite_matrix(history, snapshot_time=snapshot_time, gamma=params['gamma'], time_unit=params['time_unit'])
        Q_norm = normalize(Q)
        if z_mode == "apate":
            z_norm = get_starting_vector(known_fraud, c_map, m_map, t_map)

        elif z_mode == "apate_birank":
            if card_day_to_dict is None or merchant_day_to_dict is None:
                raise ValueError("z_mode='apate_birank' requires card_day_to_dict and merchant_day_to_dict.")
            z_norm = get_starting_vector_add_birank(
                known_fraud, c_map, m_map, t_map,
                snapshot_time=snapshot_time,
                card_day_to_dict=card_day_to_dict,
                merchant_day_to_dict=merchant_day_to_dict
            )
        else:
            raise ValueError("z_mode must be 'apate' or 'apate_birank'")

        exposure_vector = iteration(Q_norm, z_norm, alpha=0.85)

        # Build score Series for vectorized lookup
        c_scores = pd.Series({k: exposure_vector[v] for k, v in c_map.items()})
        m_scores = pd.Series({k: exposure_vector[v] for k, v in m_map.items()})
        t_scores = pd.Series({k: exposure_vector[v] for k, v in t_map.items()})

        c_col = today_txns["cc_num"].map(c_scores).fillna(0.0).to_numpy()
        m_col = today_txns["merchant"].map(m_scores).fillna(0.0).to_numpy()
        #tx_col = (c_col + m_col) / 2.0
        cc_wsum, mc_wsum, pair_latest_score = build_txscore_support_from_weights(history, row_w, t_scores)
        tx_col = compute_apate_txscore_only(today_txns, c_col, m_col, cc_wsum, mc_wsum, pair_latest_score)

        result[f"CCHScore_{window_name}"] = c_col
        result[f"MCScore_{window_name}"] = m_col
        result[f"TXScore_{window_name}"] = tx_col

    return result

In [123]:
fraudAll["_dt"] = pd.to_datetime(fraudAll["unix_time"], unit="s")
fraudAll["_day"] = fraudAll["_dt"].dt.floor("D")
u = fraudAll["unix_time"].to_numpy()
all_days = fraudAll["_day"].sort_values().unique()

tripartite_features_apate = []
for day in all_days:
    day_features = main_network_featurization(fraudAll, u, day, z_mode="apate")
    day_features["snapshot_day"] = day
    tripartite_features_apate.append(day_features)

apate_features_df = pd.concat(tripartite_features_apate, ignore_index=True)

Processing ST network features...
Converged after 10 iterations.
Processing MT network features...
Converged after 10 iterations.
Processing LT network features...
Converged after 10 iterations.
Processing ST network features...
Converged after 10 iterations.
Processing MT network features...
Converged after 10 iterations.
Processing LT network features...
Converged after 10 iterations.
Processing ST network features...
Converged after 90 iterations.
Processing MT network features...
Converged after 10 iterations.
Processing LT network features...
Converged after 10 iterations.
Processing ST network features...
Converged after 90 iterations.
Processing MT network features...
Converged after 10 iterations.
Processing LT network features...
Converged after 10 iterations.
Processing ST network features...
Converged after 90 iterations.
Processing MT network features...
Converged after 10 iterations.
Processing LT network features...
Converged after 10 iterations.
Processing ST network fea

In [124]:
tripartite_features_birank = []
for day in all_days:
    df = main_network_featurization(
        fraudAll, u, day,
        z_mode="apate_birank",
        card_day_to_dict=card_day_to_dict,
        merchant_day_to_dict=merchant_day_to_dict
    )
    df["snapshot_day"] = day
    tripartite_features_birank.append(df)
apate_birank_features_df = pd.concat(tripartite_features_birank, ignore_index=True)

Processing ST network features...
Converged after 10 iterations.
Processing MT network features...
Converged after 10 iterations.
Processing LT network features...
Converged after 10 iterations.
Processing ST network features...
Converged after 70 iterations.
Processing MT network features...
Converged after 70 iterations.
Processing LT network features...
Converged after 70 iterations.
Processing ST network features...
Converged after 70 iterations.
Processing MT network features...
Converged after 70 iterations.
Processing LT network features...
Converged after 70 iterations.
Processing ST network features...
Converged after 70 iterations.
Processing MT network features...
Converged after 70 iterations.
Processing LT network features...
Converged after 70 iterations.
Processing ST network features...
Converged after 70 iterations.
Processing MT network features...
Converged after 70 iterations.
Processing LT network features...
Converged after 70 iterations.
Processing ST network fea

In [125]:
apate_features_df

,trans_num,CCHScore_ST,MCScore_ST,TXScore_ST,CCHScore_MT,MCScore_MT,TXScore_MT,CCHScore_LT,MCScore_LT,TXScore_LT,snapshot_day
0,0b242abb623afc578575680df30655b9,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
1,1f76529f8574734946361c461b024d99,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
2,a1a22d70485983eac12b5b88dad1cf95,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
3,6b849c168bdad6f867558c3793159a81,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
4,a41d7549acf90789359a9aa5346dcb46,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
...,...,...,...,...,...,...,...,...,...,...,...
1384315,28dbfcfd6288b406e125154d26a8048b,8.675281e-06,7.022415e-03,6.825456e-03,0.000078,0.000123,0.000006,0.000244,0.001576,0.000011,2020-07-21
1384316,a057b7eecf1683d1109afd59d6478d6a,6.233391e-10,7.040730e-12,5.758263e-10,0.000066,0.000089,0.000006,0.000292,0.000216,0.000004,2020-07-21
1384317,ba8f31627906ee914d3150b3557ebe03,1.906881e-08,4.269043e-11,1.658344e-08,0.000036,0.000130,0.000007,0.000169,0.000297,0.000003,2020-07-21
1384318,014019287afe680e698414b3ca38dc09,8.184303e-06,2.164871e-16,7.065426e-06,0.000344,0.000090,0.000015,0.000280,0.000501,0.000007,2020-07-21


In [126]:
apate_birank_features_df

,trans_num,CCHScore_ST,MCScore_ST,TXScore_ST,CCHScore_MT,MCScore_MT,TXScore_MT,CCHScore_LT,MCScore_LT,TXScore_LT,snapshot_day
0,0b242abb623afc578575680df30655b9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
1,1f76529f8574734946361c461b024d99,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
2,a1a22d70485983eac12b5b88dad1cf95,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
3,6b849c168bdad6f867558c3793159a81,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
4,a41d7549acf90789359a9aa5346dcb46,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2019-01-01
...,...,...,...,...,...,...,...,...,...,...,...
1384315,28dbfcfd6288b406e125154d26a8048b,0.000793,0.000350,0.000880,0.000201,0.000252,0.000015,0.000246,0.001569,0.000011,2020-07-21
1384316,a057b7eecf1683d1109afd59d6478d6a,0.000716,0.000454,0.001105,0.000180,0.000183,0.000014,0.000294,0.000217,0.000004,2020-07-21
1384317,ba8f31627906ee914d3150b3557ebe03,0.000479,0.000396,0.000809,0.000108,0.000238,0.000015,0.000170,0.000298,0.000003,2020-07-21
1384318,014019287afe680e698414b3ca38dc09,0.000913,0.000139,0.000928,0.000403,0.000156,0.000020,0.000281,0.000499,0.000007,2020-07-21


In [131]:
# columns you want
score_cols = [
    "CCHScore_ST","MCScore_ST","TXScore_ST",
    "CCHScore_MT","MCScore_MT","TXScore_MT",
    "CCHScore_LT","MCScore_LT","TXScore_LT",
]

baseline = fraudAll[["trans_num", "_split"]].copy()

# ============================
# A) Baseline APATE features
# ============================
feat_apate = apate_features_df[["trans_num"] + score_cols].copy()

merged_apate = baseline.merge(feat_apate, on="trans_num", how="left")
merged_apate[score_cols] = merged_apate[score_cols].fillna(0.0)

apate_train = merged_apate.loc[merged_apate["_split"] == "train", ["trans_num"] + score_cols].reset_index(drop=True)
apate_test1m = merged_apate.loc[merged_apate["_split"] == "test_1m", ["trans_num"] + score_cols].reset_index(drop=True)

print("APATE merged:", merged_apate.shape, "train:", apate_train.shape, "test1m:", apate_test1m.shape)
print("APATE dup trans_num?:", feat_apate["trans_num"].duplicated().any())

apate_train.to_csv(path + "apate_features_train.csv", index=False)
apate_test1m.to_csv(path + "apate_features_test1m.csv", index=False)

# ============================
# B) APATE + BiRank features
# ============================
feat_bi = apate_birank_features_df[["trans_num"] + score_cols].copy()

merged_bi = baseline.merge(feat_bi, on="trans_num", how="left")
merged_bi[score_cols] = merged_bi[score_cols].fillna(0.0)

apate_birank_train = merged_bi.loc[merged_bi["_split"] == "train", ["trans_num"] + score_cols].reset_index(drop=True)
apate_birank_test1m = merged_bi.loc[merged_bi["_split"] == "test_1m", ["trans_num"] + score_cols].reset_index(drop=True)

print("BiRank merged:", merged_bi.shape, "train:", apate_birank_train.shape, "test1m:", apate_birank_test1m.shape)
print("BiRank dup trans_num?:", feat_bi["trans_num"].duplicated().any())

apate_birank_train.to_csv(path + "apate_birank_features_train.csv", index=False)
apate_birank_test1m.to_csv(path + "apate_birank_features_test1m.csv", index=False)

APATE merged: (1384320, 11) train: (1296675, 10) test1m: (87645, 10)
APATE dup trans_num?: False
BiRank merged: (1384320, 11) train: (1296675, 10) test1m: (87645, 10)
BiRank dup trans_num?: False


In [128]:
apate_features_csv = path + "apate_features.csv"
apate_birank_features_csv = path + "apate_features_birank.csv"

In [129]:
apate_features_df.to_csv(apate_features_csv, index=False)
apate_birank_features_df.to_csv(apate_birank_features_csv, index=False)

In [130]:
from google.colab import files
files.download(apate_features_df)
files.download(apate_birank_features_df)

TypeError: stat: path should be string, bytes, os.PathLike or integer, not DataFrame